In [1]:
# Data handling
import pandas as pd
import numpy as np

# Visualization (kept for EDA/report charts)
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

# Preprocessing utilities
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split

# Imbalanced data handling (planned for modeling stage)
from imblearn.over_sampling import SMOTE

# Classical ML models (planned for modeling stage)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier

# Evaluation metrics
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    ConfusionMatrixDisplay
)

# Deep learning stack (planned alternative baseline)
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Model persistence
import joblib

# Keep notebook output clean for teaching/demo flow.
# For debugging, consider removing this line to surface all warnings.
import warnings
warnings.filterwarnings('ignore')

# Print versions so results are reproducible for other developers.
print(f'pandas     {pd.__version__}')
print(f'numpy      {np.__version__}')
print(f'sklearn    {__import__("sklearn").__version__}')
print(f'tensorflow {tf.__version__}')
print(f'seaborn    {sns.__version__}')


Matplotlib is building the font cache; this may take a moment.


pandas     2.2.3
numpy      2.1.3
sklearn    1.6.1
tensorflow 2.20.0
seaborn    0.13.2


## Step 1: Data Profiling

### Goal
Understand the basic structure and quality of the dataset before doing any modeling.

### What this step checks
- number of rows and columns
- whether train and test have the same schema
- missing values in train and test
- class balance (`is_fraud`)
- time coverage of train/test

### How to read the output
- If schema columns do not match, stop and fix the data pipeline.
- If temporal split check fails, stop (this would cause leakage).
- If null counts are high, decide on imputation or feature removal.

### Why this matters
A fraud model is only reliable if input data is clean, consistent, and leakage-safe.

### What to report
- Train/Test size
- Fraud rate in each split
- Temporal boundary between train and test
- Whether nulls/schema checks passed


In [2]:
# ---------------------------------------------------------------------------
# Step 1: Data loading and profiling checks
# ---------------------------------------------------------------------------
# Goal: make sure train/test files are structurally valid before any mining.

TRAIN_PATH = "Data/fraudTrain.csv"
TEST_PATH  = "Data/fraudTest.csv"

# index_col=0 removes the unnamed CSV index column from Kaggle export.
df_train = pd.read_csv(TRAIN_PATH, index_col=0)
df_test  = pd.read_csv(TEST_PATH,  index_col=0)

# Basic health checks.
print(f"Train shape : {df_train.shape}")
print(f"Test  shape : {df_test.shape}")
print(f"\nFraud rate (train): {df_train['is_fraud'].mean():.4%}")
print(f"Fraud rate (test) : {df_test['is_fraud'].mean():.4%}")

# Schema consistency check: model training requires identical feature columns.
train_cols = df_train.columns.tolist()
test_cols = df_test.columns.tolist()
assert train_cols == test_cols, "Schema mismatch between train and test columns"
print(f"\nSchema check: {len(train_cols)} columns match between train and test")
print(f"Columns:\n{train_cols}")

# Missing-value checks in both splits for transparent reporting.
train_nulls = df_train.isnull().sum()
test_nulls = df_test.isnull().sum()
print(f"\nNull counts (train):\n{train_nulls[train_nulls > 0]}")
print(f"\nNull counts (test):\n{test_nulls[test_nulls > 0]}")

# Temporal holdout check: all train data must come before all test data.
# This prevents leakage from future behavior into training.
train_ts_max = pd.to_datetime(df_train["trans_date_trans_time"]).max()
test_ts_min = pd.to_datetime(df_test["trans_date_trans_time"]).min()
assert train_ts_max < test_ts_min, (
    f"Temporal split violation: max(train)={train_ts_max} is not before min(test)={test_ts_min}"
)
print(f"\nTemporal split check: max(train)={train_ts_max} < min(test)={test_ts_min}")


Train shape : (1296675, 22)
Test  shape : (555719, 22)

Fraud rate (train): 0.5789%
Fraud rate (test) : 0.3860%

Schema check: 22 columns match between train and test
Columns:
['trans_date_trans_time', 'cc_num', 'merchant', 'category', 'amt', 'first', 'last', 'gender', 'street', 'city', 'state', 'zip', 'lat', 'long', 'city_pop', 'job', 'dob', 'trans_num', 'unix_time', 'merch_lat', 'merch_long', 'is_fraud']

Null counts (train):
Series([], dtype: int64)

Null counts (test):
Series([], dtype: int64)

Temporal split check: max(train)=2020-06-21 12:13:37 < min(test)=2020-06-21 12:14:25


## Step 2: Class Imbalance Analysis

### Goal
Measure how rare fraud cases are, and quantify imbalance severity.

### What this step checks
- `fraud_count` and `non_fraud_count`
- `fraud_rate` and `non_fraud_rate`
- `imbalance_ratio = non_fraud / fraud`
- fraud-rate shift from train to test

### How to read the output
- Very large imbalance ratio means accuracy is not a useful metric.
- A train/test fraud-rate shift means threshold calibration may need adjustment.

### Why this matters
Fraud detection is a rare-event task. Imbalance drives model choice, metric choice, and threshold policy.

### What to report
- Imbalance ratio in train and test
- Fraud-rate shift (percentage points)
- Modeling implication: use class weighting and precision-recall metrics


In [4]:
# ---------------------------------------------------------------------------
# Step 2: Class imbalance analysis
# ---------------------------------------------------------------------------
# Goal: quantify rare-event imbalance and document modeling implications.

TARGET = "is_fraud"
VALID_TARGET_VALUES = {0, 1}

# Guardrail: target must stay binary in both splits.
train_target_values = set(df_train[TARGET].dropna().unique())
test_target_values = set(df_test[TARGET].dropna().unique())
assert train_target_values.issubset(VALID_TARGET_VALUES), (
    f"Unexpected target values in train: {train_target_values}"
)
assert test_target_values.issubset(VALID_TARGET_VALUES), (
    f"Unexpected target values in test: {test_target_values}"
)


def summarize_imbalance(df, split_name, target_col=TARGET):
    """Return and print core imbalance metrics for one split."""
    counts = df[target_col].value_counts().sort_index()

    # Safe .get(...) avoids KeyError if one class is missing.
    non_fraud = int(counts.get(0, 0))
    fraud = int(counts.get(1, 0))
    total = non_fraud + fraud

    fraud_rate = fraud / total if total else 0.0
    non_fraud_rate = non_fraud / total if total else 0.0
    imbalance_ratio = (non_fraud / fraud) if fraud else np.inf

    print(f"\n[{split_name}]")
    print(f"  total            : {total:,}")
    print(f"  non_fraud_count  : {non_fraud:,}")
    print(f"  fraud_count      : {fraud:,}")
    print(f"  non_fraud_rate   : {non_fraud_rate:.4%}")
    print(f"  fraud_rate       : {fraud_rate:.4%}")
    print(f"  imbalance_ratio  : {imbalance_ratio:.2f}:1 (non_fraud:fraud)")

    return {
        "split": split_name,
        "total": total,
        "non_fraud": non_fraud,
        "fraud": fraud,
        "non_fraud_rate": non_fraud_rate,
        "fraud_rate": fraud_rate,
        "imbalance_ratio": imbalance_ratio,
    }


imbalance_train = summarize_imbalance(df_train, "Train")
imbalance_test = summarize_imbalance(df_test, "Test")

# Positive value means fraud is more frequent in test than train.
fraud_rate_shift_pp = (imbalance_test["fraud_rate"] - imbalance_train["fraud_rate"]) * 100
print(f"\nFraud-rate shift (test - train): {fraud_rate_shift_pp:.4f} percentage points")

print("\nModeling implication (precision-first): prefer class-weighted training and threshold tuning over accuracy.")



[Train]
  total            : 1,296,675
  non_fraud_count  : 1,289,169
  fraud_count      : 7,506
  non_fraud_rate   : 99.4211%
  fraud_rate       : 0.5789%
  imbalance_ratio  : 171.75:1 (non_fraud:fraud)

[Test]
  total            : 555,719
  non_fraud_count  : 553,574
  fraud_count      : 2,145
  non_fraud_rate   : 99.6140%
  fraud_rate       : 0.3860%
  imbalance_ratio  : 258.08:1 (non_fraud:fraud)

Fraud-rate shift (test - train): -0.1929 percentage points

Modeling implication (precision-first): prefer class-weighted training and threshold tuning over accuracy.


## Step 3: Drift Analysis (Train vs Test)

### Goal
Check whether transaction behavior changed between training period and test period.

### What this step checks
- monthly fraud-rate trends
- numeric drift using PSI (Population Stability Index)
- categorical share shifts (`abs_delta`) for key features

### How to read the output
- PSI `< 0.10`: low drift
- PSI `0.10 - 0.25`: medium drift
- PSI `>= 0.25`: high drift
- Large categorical share deltas indicate unstable feature behavior.

### Why this matters
If data distribution changes, model quality can drop after deployment.

### What to report
- highest-drift numeric features (with PSI)
- biggest categorical shifts
- whether drift is low/medium/high overall


In [5]:
# ---------------------------------------------------------------------------
# Step 3: Drift analysis (Train vs Test)
# ---------------------------------------------------------------------------
# Goal: detect distribution changes that can reduce model reliability.

TARGET = "is_fraud"


def population_stability_index(expected, actual, bins=10, eps=1e-6):
    """Calculate PSI using quantile bins from the expected (train) distribution."""
    expected = pd.Series(expected).dropna().astype(float)
    actual = pd.Series(actual).dropna().astype(float)

    if expected.empty or actual.empty:
        return np.nan

    # Build bin edges from train quantiles (standard PSI practice).
    q = np.linspace(0, 1, bins + 1)
    breaks = expected.quantile(q).to_numpy()
    breaks = np.unique(breaks)

    # Fallback if quantiles collapse due to low variance.
    if breaks.size < 3:
        min_v = min(expected.min(), actual.min())
        max_v = max(expected.max(), actual.max())
        if min_v == max_v:
            return 0.0
        breaks = np.linspace(min_v, max_v, bins + 1)

    exp_counts, _ = np.histogram(expected, bins=breaks)
    act_counts, _ = np.histogram(actual, bins=breaks)

    exp_pct = exp_counts / max(exp_counts.sum(), 1)
    act_pct = act_counts / max(act_counts.sum(), 1)

    # Clip zeros to keep log(act/exp) numerically stable.
    exp_pct = np.clip(exp_pct, eps, None)
    act_pct = np.clip(act_pct, eps, None)

    return float(np.sum((act_pct - exp_pct) * np.log(act_pct / exp_pct)))


def psi_risk_band(psi):
    """Map PSI to common risk buckets for reporting."""
    if pd.isna(psi):
        return "unknown"
    if psi >= 0.25:
        return "high"
    if psi >= 0.10:
        return "medium"
    return "low"


# 1) Temporal drift: compare monthly fraud rates between splits.
train_month = pd.to_datetime(df_train["trans_date_trans_time"]).dt.to_period("M").astype(str)
test_month = pd.to_datetime(df_test["trans_date_trans_time"]).dt.to_period("M").astype(str)

train_monthly = (
    pd.DataFrame({"month": train_month, TARGET: df_train[TARGET]})
    .groupby("month", as_index=False)[TARGET]
    .agg(["count", "mean"])
    .reset_index()
    .rename(columns={"count": "tx_count", "mean": "fraud_rate"})
)

test_monthly = (
    pd.DataFrame({"month": test_month, TARGET: df_test[TARGET]})
    .groupby("month", as_index=False)[TARGET]
    .agg(["count", "mean"])
    .reset_index()
    .rename(columns={"count": "tx_count", "mean": "fraud_rate"})
)

print("Monthly fraud rate (train):")
print(train_monthly.tail(6).to_string(index=False))
print("\nMonthly fraud rate (test):")
print(test_monthly.head(6).to_string(index=False))


# 2) Numeric drift: PSI for core numeric fields.
numeric_candidates = ["amt", "city_pop", "lat", "long", "merch_lat", "merch_long", "unix_time"]
numeric_cols = [c for c in numeric_candidates if c in df_train.columns and c in df_test.columns]

numeric_drift_rows = []
for col in numeric_cols:
    psi = population_stability_index(df_train[col], df_test[col], bins=10)
    numeric_drift_rows.append(
        {
            "feature": col,
            "train_mean": float(df_train[col].mean()),
            "test_mean": float(df_test[col].mean()),
            "mean_delta": float(df_test[col].mean() - df_train[col].mean()),
            "train_std": float(df_train[col].std()),
            "test_std": float(df_test[col].std()),
            "psi": psi,
            "psi_risk": psi_risk_band(psi),
        }
    )

numeric_drift = pd.DataFrame(numeric_drift_rows).sort_values("psi", ascending=False)
print("\nNumeric drift summary (sorted by PSI):")
print(numeric_drift.to_string(index=False))


# 3) Categorical drift: compare category share changes.
def categorical_shift(train_df, test_df, col, top_n=10):
    """Return categories with the biggest train-vs-test share change."""
    train_share = train_df[col].astype(str).value_counts(normalize=True)
    test_share = test_df[col].astype(str).value_counts(normalize=True)

    labels = train_share.index.union(test_share.index)
    out = pd.DataFrame(
        {
            "label": labels,
            "train_share": train_share.reindex(labels, fill_value=0.0),
            "test_share": test_share.reindex(labels, fill_value=0.0),
        }
    )
    out["abs_delta"] = (out["test_share"] - out["train_share"]).abs()
    return out.sort_values("abs_delta", ascending=False).head(top_n)


categorical_cols = ["category", "gender", "state", "job"]
categorical_cols = [c for c in categorical_cols if c in df_train.columns and c in df_test.columns]

for col in categorical_cols:
    print(f"\nTop categorical share shifts for '{col}':")
    print(categorical_shift(df_train, df_test, col, top_n=10).to_string(index=False))

print("\nDrift interpretation guide: PSI < 0.10 low, 0.10-0.25 medium, >= 0.25 high.")


Monthly fraud rate (train):
 index   month  tx_count  fraud_rate
    12 2020-01     52202    0.006571
    13 2020-02     47791    0.007031
    14 2020-03     72850    0.006095
    15 2020-04     66892    0.004515
    16 2020-05     74343    0.007089
    17 2020-06     57747    0.005784

Monthly fraud rate (test):
 index   month  tx_count  fraud_rate
     0 2020-06     30058    0.004425
     1 2020-07     85848    0.003739
     2 2020-08     88759    0.004676
     3 2020-09     69533    0.004890
     4 2020-10     69348    0.005537
     5 2020-11     72635    0.004048

Numeric drift summary (sorted by PSI):
   feature    train_mean     test_mean    mean_delta    train_std     test_std       psi psi_risk
 unix_time  1.349244e+09  1.380679e+09  3.143523e+07 1.284128e+07 5.201104e+06 11.512810     high
       amt  7.035104e+01  6.939281e+01 -9.582252e-01 1.603160e+02 1.567459e+02  0.000062      low
  city_pop  8.882444e+04  8.822189e+04 -6.025526e+02 3.019564e+05 3.003909e+05  0.000046    

## Step 4: Fraud Pattern Mining (Understanding)

### Goal
Find interpretable fraud hotspots that can become useful model signals.

### What this step checks
- baseline fraud rate
- fraud lift by `category`
- fraud lift by `hour`
- fraud lift by `amt_band`
- interaction hotspots (`category x hour`)

### How to read the output
- `lift > 1`: group is riskier than average
- Use `tx_count` (support) to ignore tiny/noisy groups
- High-lift + high-support patterns are strongest candidates for features/rules.

### Why this matters
Pattern mining explains fraud behavior and helps justify feature choices in the report.

### What to report
- top high-lift categories/hours/amount bands
- top interaction hotspots
- how these patterns inform feature engineering


In [6]:
# ---------------------------------------------------------------------------
# Step 4: Fraud pattern mining
# ---------------------------------------------------------------------------
# Goal: discover interpretable fraud hotspots for feature design.

TARGET = "is_fraud"


def fraud_lift_table(df, group_col, min_tx_count=1000):
    """Compute fraud lift per group relative to the global baseline rate."""
    baseline = df[TARGET].mean()

    summary = (
        df.groupby(group_col, dropna=False)[TARGET]
        .agg(tx_count="count", fraud_count="sum", fraud_rate="mean")
        .reset_index()
    )

    # Keep only groups with enough support to avoid noisy conclusions.
    summary = summary[summary["tx_count"] >= min_tx_count].copy()
    summary["lift_vs_baseline"] = summary["fraud_rate"] / baseline
    summary = summary.sort_values("lift_vs_baseline", ascending=False)

    return summary, baseline


# Use train split only for exploratory pattern mining.
# This avoids leaking test information into feature decisions.
df_pattern = df_train.copy()
df_pattern["hour"] = pd.to_datetime(df_pattern["trans_date_trans_time"]).dt.hour

# 1) Baseline rate used as denominator for lift.
baseline_rate = df_pattern[TARGET].mean()
print(f"Baseline fraud rate (train): {baseline_rate:.4%}")

# 2) Category-level lift.
cat_lift, _ = fraud_lift_table(df_pattern, "category", min_tx_count=10000)
print("\nTop category fraud lift (train):")
print(cat_lift[["category", "tx_count", "fraud_count", "fraud_rate", "lift_vs_baseline"]].head(10).to_string(index=False))

# 3) Hour-level lift.
hour_lift, _ = fraud_lift_table(df_pattern, "hour", min_tx_count=1000)
print("\nTop hour-of-day fraud lift (train):")
print(hour_lift[["hour", "tx_count", "fraud_count", "fraud_rate", "lift_vs_baseline"]].head(10).to_string(index=False))

# 4) Amount bands provide interpretable risk buckets.
amount_bins = [0, 10, 25, 50, 100, 200, 500, 1000, np.inf]
amount_labels = ["0-10", "10-25", "25-50", "50-100", "100-200", "200-500", "500-1000", "1000+"]
df_pattern["amt_band"] = pd.cut(df_pattern["amt"], bins=amount_bins, labels=amount_labels, include_lowest=True)

amt_lift, _ = fraud_lift_table(df_pattern, "amt_band", min_tx_count=1000)
print("\nTop amount-band fraud lift (train):")
print(amt_lift[["amt_band", "tx_count", "fraud_count", "fraud_rate", "lift_vs_baseline"]].to_string(index=False))

# 5) Interaction hotspots: high-risk combinations of category and hour.
hotspots = (
    df_pattern.groupby(["category", "hour"], dropna=False)[TARGET]
    .agg(tx_count="count", fraud_count="sum", fraud_rate="mean")
    .reset_index()
)
hotspots = hotspots[hotspots["tx_count"] >= 500].copy()
hotspots["lift_vs_baseline"] = hotspots["fraud_rate"] / baseline_rate
hotspots = hotspots.sort_values(["lift_vs_baseline", "fraud_count"], ascending=[False, False])

print("\nTop interaction hotspots (category x hour, train):")
print(hotspots[["category", "hour", "tx_count", "fraud_count", "fraud_rate", "lift_vs_baseline"]].head(15).to_string(index=False))

print("\nInterpretation note: prioritize high-lift groups with enough support for feature engineering and threshold policy.")


Baseline fraud rate (train): 0.5789%

Top category fraud lift (train):
     category  tx_count  fraud_count  fraud_rate  lift_vs_baseline
 shopping_net     97543         1713    0.017561          3.033778
     misc_net     63287          915    0.014458          2.497636
  grocery_pos    123638         1743    0.014098          2.435387
 shopping_pos    116672          843    0.007225          1.248198
gas_transport    131659          618    0.004694          0.810887
     misc_pos     79655          250    0.003139          0.542188
  grocery_net     45452          134    0.002948          0.509301
       travel     40507          116    0.002864          0.494710
entertainment     94014          233    0.002478          0.428140
personal_care     90758          220    0.002424          0.418755

Top hour-of-day fraud lift (train):
 hour  tx_count  fraud_count  fraud_rate  lift_vs_baseline
   22     66982         1931    0.028829          4.980200
   23     67104         1904    0.028

## Step 5: Leakage and Proxy-Risk Analysis

### Goal
Identify raw columns that may create unrealistic model performance or fairness risks.

### What this step checks
- direct identifiers (`trans_num`, `cc_num`, names, street)
- raw time fields that can leak sequence artifacts
- personal/location proxy features
- near-unique high-cardinality fields

### How to read the output
- `drop_now`: do not use raw feature in final model
- `engineer_then_drop_raw`: create safe derived features, then remove raw column
- `keep_with_monitor`: allowed, but track drift/performance impact
- `keep`: low-risk feature

### Why this matters
Leakage-safe features are required for honest evaluation and stable production behavior.

### What to report
- feature risk table (risk + reason + recommendation)
- final keep/drop/watch decisions
- identifier overlap audit notes (if available)


In [7]:
# ---------------------------------------------------------------------------
# Step 5: Leakage and proxy-risk analysis
# ---------------------------------------------------------------------------
# Goal: identify columns that can create unrealistic model performance.

TARGET = "is_fraud"

# Risk dictionaries based on domain knowledge.
DIRECT_IDENTIFIER_COLS = {"trans_num", "cc_num", "first", "last", "street"}
TEMPORAL_RAW_COLS = {"trans_date_trans_time", "unix_time"}
PERSONAL_PROXY_COLS = {"dob", "zip", "city", "state", "job"}


def classify_feature_risk(col_name, unique_ratio):
    """Assign risk level + recommended action for one raw feature."""
    if col_name in DIRECT_IDENTIFIER_COLS:
        return "high", "drop_now", "direct_identifier"
    if col_name in TEMPORAL_RAW_COLS:
        return "high", "engineer_then_drop_raw", "raw_time_granularity"
    if col_name in PERSONAL_PROXY_COLS:
        return "medium", "keep_with_monitor", "personal_or_location_proxy"
    if unique_ratio >= 0.95:
        return "medium", "keep_with_monitor", "near_unique_high_cardinality"
    return "low", "keep", "no_major_risk"


feature_rows = []
feature_cols = [c for c in df_train.columns if c != TARGET]

for col in feature_cols:
    s = df_train[col]
    non_null = s.notna().sum()
    nunique = int(s.nunique(dropna=True))
    unique_ratio = (nunique / non_null) if non_null else 0.0

    risk_level, recommendation, reason = classify_feature_risk(col, unique_ratio)

    feature_rows.append(
        {
            "feature": col,
            "dtype": str(s.dtype),
            "non_null": int(non_null),
            "nunique": nunique,
            "unique_ratio": float(unique_ratio),
            "risk_level": risk_level,
            "reason": reason,
            "recommendation": recommendation,
        }
    )

risk_df = pd.DataFrame(feature_rows)

# Sort by risk first so report readers see the most important issues first.
risk_order = {"high": 0, "medium": 1, "low": 2}
risk_df["_risk_order"] = risk_df["risk_level"].map(risk_order)
risk_df = risk_df.sort_values(["_risk_order", "unique_ratio"], ascending=[True, False]).drop(columns=["_risk_order"])

print("Feature risk assessment:")
print(risk_df.to_string(index=False))


# Build action buckets for implementation planning.
drop_now = risk_df.loc[risk_df["recommendation"] == "drop_now", "feature"].tolist()
engineer_then_drop_raw = risk_df.loc[risk_df["recommendation"] == "engineer_then_drop_raw", "feature"].tolist()
keep_with_monitor = risk_df.loc[risk_df["recommendation"] == "keep_with_monitor", "feature"].tolist()
keep = risk_df.loc[risk_df["recommendation"] == "keep", "feature"].tolist()

print("\nRecommended feature actions:")
print(f"  drop_now               : {drop_now}")
print(f"  engineer_then_drop_raw : {engineer_then_drop_raw}")
print(f"  keep_with_monitor      : {keep_with_monitor}")
print(f"  keep                   : {keep}")

# Optional overlap checks for audit transparency.
if "trans_num" in df_train.columns and "trans_num" in df_test.columns:
    overlap_trans_num = len(set(df_train["trans_num"]).intersection(set(df_test["trans_num"])))
    print(f"\nIdentifier overlap check: trans_num overlap train/test = {overlap_trans_num}")

if "cc_num" in df_train.columns and "cc_num" in df_test.columns:
    overlap_cc_num = len(set(df_train["cc_num"]).intersection(set(df_test["cc_num"])))
    print(f"Identifier overlap check: cc_num overlap train/test = {overlap_cc_num}")

print("\nGuideline: high-risk columns should not enter the final model as raw features.")


Feature risk assessment:
              feature   dtype  non_null  nunique  unique_ratio risk_level                       reason         recommendation
            trans_num  object   1296675  1296675      1.000000       high            direct_identifier               drop_now
            unix_time   int64   1296675  1274823      0.983148       high         raw_time_granularity engineer_then_drop_raw
trans_date_trans_time  object   1296675  1274791      0.983123       high         raw_time_granularity engineer_then_drop_raw
               cc_num   int64   1296675      983      0.000758       high            direct_identifier               drop_now
               street  object   1296675      983      0.000758       high            direct_identifier               drop_now
                 last  object   1296675      481      0.000371       high            direct_identifier               drop_now
                first  object   1296675      352      0.000271       high            direct_i

## Step 6: Feature Strategy Decision (Precision-First)

### Goal
Convert analysis findings (Steps 1-5) into a final, implementation-ready feature policy.

### What this step decides
- which raw columns are dropped immediately
- which raw columns are used to create derived features, then dropped
- which columns are kept as model inputs
- categorical encoding and scaling policy

### How to read the output
- The feature decision table is the source of truth for preprocessing.
- `engineer_then_drop_raw` means: keep temporarily only to build new features.
- `keep_raw` means: column can remain in the model input after preprocessing.

### Why this matters
A clear feature policy avoids leakage, keeps modeling reproducible, and makes implementation easier for other developers.

### What to report
- final keep/drop/engineer decisions per feature
- final derived feature list
- categorical feature list
- preprocessing policy (encoding, scaling, imbalance handling)


In [8]:
# ---------------------------------------------------------------------------
# Step 6: Feature strategy decision (precision-first)
# ---------------------------------------------------------------------------
# Goal: create one clear feature policy that downstream cells can reuse.

TARGET = "is_fraud"

# 1) Final decisions from previous analysis steps.
# - drop_now: high leakage / direct identifier fields.
# - engineer_then_drop_raw: raw fields used only to create safer derived features.
# - keep_raw: allowed as model inputs after preprocessing.
DROP_NOW_COLS = [
    "cc_num", "first", "last", "street", "trans_num", "zip", "city", "state"
]

DERIVED_FEATURE_MAP = {
    "trans_date_trans_time": ["hour", "day_of_week", "month"],
    "dob": ["age"],
    "lat": ["distance_km"],
    "long": ["distance_km"],
    "merch_lat": ["distance_km"],
    "merch_long": ["distance_km"],
    "unix_time": []
}

KEEP_RAW_COLS = ["merchant", "category", "amt", "gender", "city_pop", "job"]

# 2) Downstream preprocessing policy.
CATEGORICAL_FEATURES = ["merchant", "category", "gender", "job"]
NUMERIC_FEATURES_AFTER_ENGINEERING = ["amt", "city_pop", "hour", "day_of_week", "month", "age", "distance_km"]
AGE_REFERENCE_DATE = pd.Timestamp("2020-06-21")

# For feature engineering cell: after deriving new features, drop these raw columns.
RAW_DROP_AFTER_ENGINEERING = sorted(set(DROP_NOW_COLS + list(DERIVED_FEATURE_MAP.keys())))

PREPROCESSING_POLICY = {
    "categorical_encoding": "LabelEncoder (fit on train only, unseen in test -> -1)",
    "scaling": "StandardScaler (fit on train only)",
    "imbalance_default": "class_weight",
    "objective": "high_precision"
}

# 3) Build a transparent per-feature decision table for reporting.
feature_rows = []
for col in df_train.columns:
    if col == TARGET:
        action = "target"
        reason = "prediction_target"
        derived_outputs = []
    elif col in DROP_NOW_COLS:
        action = "drop_now"
        reason = "identifier_or_high_proxy_risk"
        derived_outputs = []
    elif col in DERIVED_FEATURE_MAP:
        action = "engineer_then_drop_raw"
        reason = "convert_raw_signal_into_safer_feature"
        derived_outputs = DERIVED_FEATURE_MAP[col]
    elif col in KEEP_RAW_COLS:
        action = "keep_raw"
        reason = "useful_predictive_signal"
        derived_outputs = []
    else:
        action = "review_needed"
        reason = "not_in_explicit_policy"
        derived_outputs = []

    feature_rows.append(
        {
            "feature": col,
            "action": action,
            "reason": reason,
            "derived_outputs": ", ".join(derived_outputs)
        }
    )

feature_strategy_df = pd.DataFrame(feature_rows).sort_values(["action", "feature"]).reset_index(drop=True)

# 4) Sanity checks for student-friendly debugging.
missing_keep = [c for c in KEEP_RAW_COLS if c not in df_train.columns]
missing_derived = [c for c in DERIVED_FEATURE_MAP.keys() if c not in df_train.columns]
assert not missing_keep, f"Missing keep columns in train data: {missing_keep}"
assert not missing_derived, f"Missing raw columns for derived features: {missing_derived}"

print("Feature strategy table:")
print(feature_strategy_df.to_string(index=False))

print("\nFinal policy summary:")
print(f"  drop_now_cols                : {DROP_NOW_COLS}")
print(f"  engineer_then_drop_raw_cols  : {list(DERIVED_FEATURE_MAP.keys())}")
print(f"  keep_raw_cols                : {KEEP_RAW_COLS}")
print(f"  derived_features             : {sorted(set(sum(DERIVED_FEATURE_MAP.values(), [])))}")
print(f"  categorical_features         : {CATEGORICAL_FEATURES}")
print(f"  numeric_features             : {NUMERIC_FEATURES_AFTER_ENGINEERING}")
print(f"  preprocessing_policy         : {PREPROCESSING_POLICY}")


Feature strategy table:
              feature                 action                                reason          derived_outputs
               cc_num               drop_now         identifier_or_high_proxy_risk                         
                 city               drop_now         identifier_or_high_proxy_risk                         
                first               drop_now         identifier_or_high_proxy_risk                         
                 last               drop_now         identifier_or_high_proxy_risk                         
                state               drop_now         identifier_or_high_proxy_risk                         
               street               drop_now         identifier_or_high_proxy_risk                         
            trans_num               drop_now         identifier_or_high_proxy_risk                         
                  zip               drop_now         identifier_or_high_proxy_risk                         
    

In [9]:
# ---------------------------------------------------------------------------
# Feature engineering
# ---------------------------------------------------------------------------
# Goal: transform raw transaction data into model-friendly predictive features.


def haversine_km(lat1, lon1, lat2, lon2):
    """Vectorized Haversine distance in kilometers."""
    R = 6371.0
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlam = np.radians(lon2 - lon1)
    a = np.sin(dphi / 2)**2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlam / 2)**2
    return R * 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))


def engineer_features(df):
    """Create engineered features and return a transformed copy."""
    df = df.copy()

    # Time-based behavior features from raw transaction timestamp.
    ts = pd.to_datetime(df["trans_date_trans_time"])
    df["hour"] = ts.dt.hour
    df["day_of_week"] = ts.dt.dayofweek  # 0=Monday, 6=Sunday
    df["month"] = ts.dt.month

    # Customer age at reference date. Step 6 can override this date if needed.
    reference_date = globals().get("AGE_REFERENCE_DATE", pd.Timestamp("2020-06-21"))
    df["age"] = ((reference_date - pd.to_datetime(df["dob"])).dt.days / 365.25).round(1)

    # Cardholder-to-merchant geographic distance.
    df["distance_km"] = haversine_km(
        df["lat"], df["long"], df["merch_lat"], df["merch_long"]
    )

    # Prefer Step 6 drop policy; fallback keeps this cell runnable standalone.
    default_cols_to_drop = [
        "trans_date_trans_time", "cc_num", "first", "last",
        "street", "city", "state", "zip",
        "lat", "long", "merch_lat", "merch_long",
        "trans_num", "unix_time", "dob",
    ]
    cols_to_drop = globals().get("RAW_DROP_AFTER_ENGINEERING", default_cols_to_drop)
    return df.drop(columns=cols_to_drop)


# Apply identical feature logic to both splits.
df_train_eng = engineer_features(df_train)
df_test_eng = engineer_features(df_test)

print(f"Train after engineering: {df_train_eng.shape}")
print(f"Columns: {df_train_eng.columns.tolist()}")
print(df_train_eng.dtypes)


Train after engineering: (1296675, 12)
Columns: ['merchant', 'category', 'amt', 'gender', 'city_pop', 'job', 'is_fraud', 'hour', 'day_of_week', 'month', 'age', 'distance_km']
merchant        object
category        object
amt            float64
gender          object
city_pop         int64
job             object
is_fraud         int64
hour             int32
day_of_week      int32
month            int32
age            float64
distance_km    float64
dtype: object


In [10]:
# ---------------------------------------------------------------------------
# Encoding, scaling, and NumPy export
# ---------------------------------------------------------------------------
# Goal: convert engineered DataFrames into model-ready matrices.

# Prefer Step 6 policy values; fallback keeps this cell runnable standalone.
CATEGORICAL_COLS = globals().get("CATEGORICAL_FEATURES", ["merchant", "category", "gender", "job"])
TARGET = "is_fraud"

# Fit encoders on train only to avoid test leakage.
# Unseen categories in test are mapped to -1 as a safe fallback.
encoders = {}
df_train_enc = df_train_eng.copy()
df_test_enc = df_test_eng.copy()

for col in CATEGORICAL_COLS:
    le = LabelEncoder()
    le.fit(df_train_enc[col])
    encoders[col] = le

    # Normal transform on training categories.
    df_train_enc[col] = le.transform(df_train_enc[col])

    # Map test categories using train mapping; unknown labels -> -1.
    mapping = dict(zip(le.classes_, le.transform(le.classes_)))
    df_test_enc[col] = df_test_enc[col].map(mapping).fillna(-1).astype(int)

    unseen = (df_test_enc[col] == -1).sum()
    print(f"  {col:12s}: {len(le.classes_)} train classes | {unseen} unseen -> -1")

# Split predictors and target.
FEATURE_COLS = [c for c in df_train_enc.columns if c != TARGET]

X_train = df_train_enc[FEATURE_COLS]
y_train = df_train_enc[TARGET]
X_test = df_test_enc[FEATURE_COLS]
y_test = df_test_enc[TARGET]

print(f"\nX_train: {X_train.shape}  |  y_train: {y_train.shape}")
print(f"X_test : {X_test.shape}   |  y_test : {y_test.shape}")
print(f"Features ({len(FEATURE_COLS)}): {FEATURE_COLS}")

# Standardize numeric scale (fit on train, apply to test).
scaler = StandardScaler()
X_train_np = scaler.fit_transform(X_train)
X_test_np = scaler.transform(X_test)
y_train_np = y_train.to_numpy(dtype=np.int32)
y_test_np = y_test.to_numpy(dtype=np.int32)

# Final audit of model-ready arrays.
print("\n=== Final NumPy arrays ===")
print(f"X_train_np : {X_train_np.shape}  dtype={X_train_np.dtype}")
print(f"X_test_np  : {X_test_np.shape}   dtype={X_test_np.dtype}")
print(f"y_train_np : {y_train_np.shape}  fraud={y_train_np.sum()} ({y_train_np.mean():.4%})")
print(f"y_test_np  : {y_test_np.shape}   fraud={y_test_np.sum()} ({y_test_np.mean():.4%})")

# Keep unscaled DataFrames for EDA/debugging if needed.


  merchant    : 693 train classes | 0 unseen -> -1
  category    : 14 train classes | 0 unseen -> -1
  gender      : 2 train classes | 0 unseen -> -1
  job         : 494 train classes | 30 unseen -> -1

X_train: (1296675, 11)  |  y_train: (1296675,)
X_test : (555719, 11)   |  y_test : (555719,)
Features (11): ['merchant', 'category', 'amt', 'gender', 'city_pop', 'job', 'hour', 'day_of_week', 'month', 'age', 'distance_km']

=== Final NumPy arrays ===
X_train_np : (1296675, 11)  dtype=float64
X_test_np  : (555719, 11)   dtype=float64
y_train_np : (1296675,)  fraud=7506 (0.5789%)
y_test_np  : (555719,)   fraud=2145 (0.3860%)


## Step 7: Baseline Modeling (Precision-First)

### Goal
Train simple, reproducible baseline models and compare them with fraud-appropriate metrics.

### Models currently involved
- `log_reg_balanced` (`LogisticRegression`): linear baseline with `class_weight="balanced"`
- `rf_balanced` (`RandomForestClassifier`): non-linear tree ensemble with `class_weight="balanced_subsample"`

### Model schematics (planned implementation flow)
`Raw data` -> `Step 6 feature policy` -> `Feature engineering` -> `Encoding + scaling` -> `Model` -> `Fraud probability score` -> `Threshold (Step 8)` -> `Fraud alert / no alert`

### Why these two baselines first
- Logistic Regression gives a simple, interpretable benchmark.
- Random Forest captures non-linear patterns and interactions.
- Together, they provide a strong minimum benchmark before trying more complex models.

### What this step does
- trains class-weighted baseline models
- evaluates ranking quality (`ROC-AUC`, `PR-AUC`)
- evaluates alert quality at default threshold (`precision`, `recall`, alert rate)
- produces a comparison table for model selection

### How to read the output
- Higher `PR-AUC` is usually more important than accuracy in fraud tasks.
- Higher `precision_at_0_5` means fewer false alerts.
- `alert_rate_at_0_5` shows operational workload impact.

### Why this matters
This creates a transparent starting point before threshold tuning (Step 8) and final recommendation (Step 9).

### What to report
- baseline model comparison table
- best candidate by `PR-AUC`
- tradeoff notes for precision vs recall
- model schematic used for implementation


In [11]:
# ---------------------------------------------------------------------------
# Step 7: Baseline modeling (precision-first)
# ---------------------------------------------------------------------------
# Goal: train simple baseline models and compare them with fraud-focused metrics.
#
# Models used in this step:
# 1) Logistic Regression (class-weighted)
#    - Structure: linear decision boundary in transformed feature space.
#    - Role: interpretable baseline and sanity check.
#
# 2) Random Forest (class-weighted)
#    - Structure: ensemble of decision trees with majority/probability voting.
#    - Role: captures non-linear interactions and serves as stronger baseline.
#
# Planned pipeline schematic:
# Raw/engineered features -> model.predict_proba(...) -> fraud score
# fraud score + threshold (Step 8) -> alert decision

from sklearn.metrics import average_precision_score, precision_score, recall_score, f1_score

# Reproducibility seed.
RANDOM_STATE = 42

# Baseline model set (class-weighted for severe class imbalance).
baseline_models = {
    "log_reg_balanced": LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=RANDOM_STATE
    ),
    "rf_balanced": RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced_subsample",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
}

# Student-friendly model summary for report/debug output.
MODEL_SUMMARY = {
    "log_reg_balanced": {
        "family": "linear model",
        "core_structure": "single linear decision function",
        "strength": "interpretable and fast",
    },
    "rf_balanced": {
        "family": "tree ensemble",
        "core_structure": "many decision trees aggregated",
        "strength": "captures non-linear patterns",
    },
}

print("Models involved in Step 7:")
for name, info in MODEL_SUMMARY.items():
    print(f"  - {name}: {info['family']} | structure={info['core_structure']} | strength={info['strength']}")


def evaluate_binary_model(name, model, X_tr, y_tr, X_te, y_te, threshold=0.5):
    """Fit one model and return fraud-relevant evaluation metrics."""
    model.fit(X_tr, y_tr)

    # Use fraud probability for ranking and threshold analysis.
    y_score = model.predict_proba(X_te)[:, 1]
    y_pred = (y_score >= threshold).astype(int)

    roc_auc = roc_auc_score(y_te, y_score)
    pr_auc = average_precision_score(y_te, y_score)
    precision = precision_score(y_te, y_pred, zero_division=0)
    recall = recall_score(y_te, y_pred, zero_division=0)
    f1 = f1_score(y_te, y_pred, zero_division=0)

    alert_count = int(y_pred.sum())
    alert_rate = float(alert_count / len(y_pred))

    metrics = {
        "model": name,
        "roc_auc": float(roc_auc),
        "pr_auc": float(pr_auc),
        "precision_at_0_5": float(precision),
        "recall_at_0_5": float(recall),
        "f1_at_0_5": float(f1),
        "alert_count_at_0_5": alert_count,
        "alert_rate_at_0_5": alert_rate,
    }

    artifacts = {
        "model": model,
        "y_score_test": y_score,
        "y_pred_test_at_0_5": y_pred,
    }

    return metrics, artifacts


baseline_metrics = []
baseline_artifacts = {}

for model_name, model in baseline_models.items():
    print(f"\nTraining baseline model: {model_name}")

    metrics, artifacts = evaluate_binary_model(
        name=model_name,
        model=model,
        X_tr=X_train_np,
        y_tr=y_train_np,
        X_te=X_test_np,
        y_te=y_test_np,
        threshold=0.5,
    )

    baseline_metrics.append(metrics)
    baseline_artifacts[model_name] = artifacts


# Create a clean comparison table for report use.
baseline_results_df = pd.DataFrame(baseline_metrics).sort_values("pr_auc", ascending=False).reset_index(drop=True)

print("\nBaseline model comparison (sorted by PR-AUC):")
print(baseline_results_df.to_string(index=False))

# Candidate model for Step 8 threshold tuning.
best_model_name = baseline_results_df.loc[0, "model"]
best_model = baseline_artifacts[best_model_name]["model"]
best_model_test_scores = baseline_artifacts[best_model_name]["y_score_test"]

print(f"\nSelected baseline candidate for Step 8: {best_model_name}")
print("Selection rule: highest PR-AUC, then inspect precision/alert-rate tradeoff.")


Models involved in Step 7:
  - log_reg_balanced: linear model | structure=single linear decision function | strength=interpretable and fast
  - rf_balanced: tree ensemble | structure=many decision trees aggregated | strength=captures non-linear patterns

Training baseline model: log_reg_balanced

Training baseline model: rf_balanced

Baseline model comparison (sorted by PR-AUC):
           model  roc_auc   pr_auc  precision_at_0_5  recall_at_0_5  f1_at_0_5  alert_count_at_0_5  alert_rate_at_0_5
     rf_balanced 0.987662 0.883456          0.957617       0.726807   0.826398                1628           0.002930
log_reg_balanced 0.851319 0.145177          0.075841       0.741259   0.137603               20965           0.037726

Selected baseline candidate for Step 8: rf_balanced
Selection rule: highest PR-AUC, then inspect precision/alert-rate tradeoff.
